# 🎯 [DAT06_M02] 다양한 양식의 엑셀/CSV 데이터 자동 파싱, 정합성 검증 및 전처리 파이프라인 구축
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m02_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M02 |
| **미션 주제** | 다양한 양식의 엑셀/CSV 데이터 자동 파싱, 정합성 검증 및 전처리 파이프라인 구축 |
| **소요 시간** | 약 2.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 다양한 양식의 엑셀/CSV 데이터 자동 파싱, 정합성 검증 및 전처리 파이프라인 구축
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

엑셀/CSV 데이터 자동 파싱, 정합성 검증 및 전처리 파이프라인을 구축한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **CSV, 엑셀 내보내기(탭 구분), JSON, 공문 텍스트 형식의 결산 자료를 자동 파싱하고 정합성을 검증하는 전처리 파이프라인을 구축한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **서로 다른 서식의 가상 결산 자료 4종을 메모리 안의 문자열로 만들고 형식별 파서로 표준 레코드로 바꾼다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/connectors/dat06_data_loader.py 및 스키마 정의서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 국방 서식/양식 변경에 대한 유연한 확장성

#### 미션 PASS 조건

- 정량 기준: 지원 파일 포맷(CSV, Excel, JSON 등) 파싱 성공률 100%


- 진행 기준: 데이터 로더 및 스키마 유효성 검사 모듈 완성


- 검증 방법: 다양한 형태의 샘플 군수 문서 대상 파싱 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""CSV, 엑셀 내보내기(탭 구분), JSON, 공문 텍스트 4가지 형식의 가상 결산 자료를 메모리 안에서 자동 파싱하고 머리글 별칭 대응·정합성 검증·전처리를 거쳐 표준 원장으로 합치는 파이프라인을 구축한다."""
import csv
import io
import json
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

TRUTH = [r for r in make_ledger(2) if r["month"] == "2026-08"]
FIELDS = ["unit", "cat", "item", "open", "recv", "issue", "ret", "close"]
ALIAS = {"unit": ["unit", "부대", "부대명", "제출부대"], "cat": ["cat", "구분", "분류"], "item": ["item", "품목", "품목명"],
         "open": ["open", "기초", "기초재고"], "recv": ["recv", "수령", "입고"], "issue": ["issue", "불출", "소모", "사용"],
         "ret": ["ret", "반납"], "close": ["close", "기말", "기말재고"]}
H2F = {h: f for f, hs in ALIAS.items() for h in hs}
by_unit = lambda a, b: [r for r in TRUTH if UNITS[a] <= r["unit"] <= UNITS[b]]

# --- 가상 원본 4종(문자열) 생성: 형식마다 서식이 다르다 ---
def mk_csv(rows):
    return "\n".join(["부대,구분,품목,기초,수령,불출,반납,기말"] + [",".join(str(r[f]) for f in FIELDS) for r in rows])
def mk_tab(rows):        # 엑셀에서 '탭 구분 텍스트'로 내보낸 형태, 머리글이 양식 개정으로 바뀜
    return "\n".join(["제출부대\t분류\t품목명\t기초재고\t입고\t소모\t반납\t기말재고"] + ["\t".join(str(r[f]) for f in FIELDS) for r in rows])
def mk_json(rows):
    return json.dumps({"form": "결산서", "rows": [{f: r[f] for f in FIELDS} for r in rows]}, ensure_ascii=False)
def mk_memo(rows):       # 공문 본문 형태 비정형 텍스트
    return "\n".join(f"{i + 1}. {r['unit']} {r['cat']} {r['item']}: 기초 {r['open']}, 수령 {r['recv']}, 불출 {r['issue']}, 반납 {r['ret']}, 기말 {r['close']}"
                     for i, r in enumerate(rows))
RAW = {"csv": mk_csv(by_unit(0, 2)), "excel_tab": mk_tab(by_unit(3, 5)), "json": mk_json(by_unit(6, 8)), "memo": mk_memo(by_unit(9, 11))}
RAW["csv"] += "\n가상99대대,물자불출,급식류,10,0,3,0,7"   # 존재하지 않는 부대 코드(보류 대상)
RAW["memo"] += "\n99. 가상10대대 탄약정산 훈련탄A: 기초 50, 수령 10, 불출 abc, 반납 0, 기말 60"   # 숫자 아님(보류 대상)

# --- 파서: 모두 표준 레코드(dict) 리스트로 변환 ---
def norm(row):
    out = {}
    for k, v in row.items():
        f = H2F.get(str(k).strip())
        if f:
            out[f] = v.strip() if isinstance(v, str) else v
    return out
def parse_csv(s, delim=","):
    return [norm(r) for r in csv.DictReader(io.StringIO(s), delimiter=delim)]
def parse_json(s):
    return [norm(r) for r in json.loads(s)["rows"]]
def parse_memo(s):
    pat = re.compile(r"\d+\.\s*(\S+)\s+(\S+)\s+(\S+):\s*기초\s*(\S+),\s*수령\s*(\S+),\s*불출\s*(\S+),\s*반납\s*(\S+),\s*기말\s*(\S+)")
    return [dict(zip(FIELDS, m.groups())) for m in map(pat.match, s.splitlines()) if m]
PARSERS = {"csv": parse_csv, "excel_tab": lambda s: parse_csv(s, "\t"), "json": parse_json, "memo": parse_memo}

# --- 전처리·정합성 검증: 형 변환, 코드 확인, 산식 대조. 오류 레코드는 반영 보류 ---
known_items = {it for v in ITEMS.values() for it in v}
def clean(r, fmt):
    try:
        for f in ("open", "recv", "issue", "ret", "close"):
            r[f] = int(r[f])
    except (ValueError, KeyError):
        return None, f"[{fmt}] 숫자 변환 실패: {r.get('unit')} {r.get('item')}"
    if r["unit"] not in UNITS:
        return None, f"[{fmt}] 존재하지 않는 부대 코드 {r['unit']}"
    if r["item"] not in known_items:
        return None, f"[{fmt}] 알 수 없는 품목 {r['item']}"
    if r["open"] + r["recv"] + r["ret"] - r["issue"] != r["close"]:
        return None, f"[{fmt}] 재고 산식 불일치 {r['unit']} {r['item']}"
    return r, None

good, hold, stat = [], [], {}
for fmt, text in RAW.items():
    try:
        recs = PARSERS[fmt](text)
        stat[fmt] = {"parse_ok": True, "n": len(recs)}
    except Exception as ex:
        recs, stat[fmt] = [], {"parse_ok": False, "n": 0, "err": repr(ex)[:50]}
    for r in recs:
        c, why = clean(r, fmt)
        (good if c else hold).append(c or why)
df = pd.DataFrame(good, columns=FIELDS).sort_values(["unit", "item"]).reset_index(drop=True)

print("[형식별 파싱 결과] [교육용가상자료]")
for fmt, s in stat.items():
    print(f" - {fmt:<9} 파싱 {'성공' if s['parse_ok'] else '실패'}, 읽은 레코드 {s['n']}건")
rate = sum(s["parse_ok"] for s in stat.values()) / len(stat)
print(f"파싱 성공률 {rate:.0%}({len(stat)}개 형식), 정상 반영 {len(df)}건, 반영 보류 {len(hold)}건")
for w in hold:
    print("   보류:", w)
truth = pd.DataFrame(by_unit(0, 11), columns=FIELDS).sort_values(["unit", "item"]).reset_index(drop=True)
same = df.equals(truth.astype(df.dtypes.to_dict()))
print(f"원본 대비 값 일치: {'전부 일치' if same else '불일치'} ({len(df)}행 = 12개 부대 x 11품목 = {12 * 11}행)")
print("표준 원장 미리보기(앞 4행):")
print(df.head(4).to_string(index=False))
print(f"\n목표(4개 형식 파싱 성공률 100%, 양식 변경 대응, 오류 레코드 보류): {'달성' if rate == 1 and same and len(hold) == 2 else '미달'}")
print("새 서식 대응은 ALIAS 사전에 머리글 별칭만 추가하면 된다(코드 수정 불필요).")


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[형식별 파싱 결과] [교육용가상자료]
 - csv       파싱 성공, 읽은 레코드 34건
 - excel_tab 파싱 성공, 읽은 레코드 33건
 - json      파싱 성공, 읽은 레코드 33건
 - memo      파싱 성공, 읽은 레코드 34건
파싱 성공률 100%(4개 형식), 정상 반영 132건, 반영 보류 2건
   보류: [csv] 존재하지 않는 부대 코드 가상99대대
   보류: [memo] 숫자 변환 실패: 가상10대대 훈련탄A
원본 대비 값 일치: 전부 일치 (132행 = 12개 부대 x 11품목 = 132행)
표준 원장 미리보기(앞 4행):
  unit  cat item  open  recv  issue  ret  close
가상01대대 물자불출  급식류   402    51     39    3    417
가상01대대 탄약정산  모의탄   943   147    117   11    984
가상01대대 수리부속 베어링류   261    34    142    1    154
가상01대대 물자불출   유류  1279   257    108    5   1433

목표(4개 형식 파싱 성공률 100%, 양식 변경 대응, 오류 레코드 보류): 달성
새 서식 대응은 ALIAS 사전에 머리글 별칭만 추가하면 된다(코드 수정 불필요).
```
</details>
